In [0]:
%run ./P5_00_Config

In [0]:
%pip install prophet

from prophet import Prophet
import mlflow
import pandas as pd

In [0]:
df_silver = spark.table(SILVER_TABLE)

df_automotive = df_silver.filter(df_silver.family == 'AUTOMOTIVE')

df_automotive = df_automotive.toPandas()

df_automotive["date"] = pd.to_datetime(df_automotive["date"])

In [0]:
df_prophet = df_automotive[["date", "sales"]]
df_prophet = df_prophet.rename(columns={"date": "ds", "sales": "y"})

In [0]:
with mlflow.start_run(run_name="prophet_automotive"):

    #Train the model
    model = Prophet()
    model.fit(df_prophet)

    #Generate future dates
    future = model.make_future_dataframe(periods=FORECAST_HORIZON)

    #Generate predictions
    forecast = model.predict(future)

    #Log parameters to MLflow
    mlflow.log_param("horizon", FORECAST_HORIZON)
    mlflow.log_param("family", "AUTOMOTIVE")

    # Calculate MAE on training data
    df_eval = forecast[["ds", "yhat"]].merge(df_prophet, on="ds")
    mae = (df_eval["y"] - df_eval["yhat"]).abs().mean()
    
    # Log metric to MLflow
    mlflow.log_metric("mae", mae)
    print(f"MAE: {mae:.2f}")

In [0]:
df_gold = forecast[["ds", "yhat", "yhat_upper", "yhat_lower"]].copy()
df_gold["family"] = "AUTOMOTIVE"
df_gold["model"] = "prophet"

df_gold = spark.createDataFrame(df_gold)
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE)